In [ ]:
# %uv init

In [ ]:
from fastapi import FastAPI
import uvicorn
import nest_asyncio

nest_asyncio.apply()

# Custom paths for OpenAPI spec and documentation UIs
app = FastAPI(
    openapi_url="/api/v1/openapi.json",  # Customizes raw spec path
    docs_url="/documentation",          # Customizes Swagger UI path
    redoc_url="/alternative-docs"        # Customizes ReDoc path
)

@app.get("/")
def read_root():
   return {"message": "Welcome to FastAPI!"}


@app.get("/items/{item_id}")
def read_item(item_id: int):
    return {"item_id": item_id}


@app.get("/search")
def search_object(keyword: int | None = None , pages: int | None = None, title: str | None = None) -> dict:

    return {
        "keyword" : keyword,
        "pages": pages,
        "title" : title
    }



# why we are using the pydantic


## Request Bodies with Pydantic Models

### 1. Data Parsing and Type Coercion
HTTP request payloads (JSON strings, query parameters, form data) arrive as raw text. Before processing them in business logic, developers must convert strings to proper Python types (integers, datetimes, booleans, UUIDs, etc.).

Pydantic automatically **parses and coerces** compatible data types without forcing rigid types on the client:
* If a model expects an `int` age and receives `"25"` (string), Pydantic safely converts it to integer `25`.
* If a model expects a `datetime` and receives `"2026-08-20T14:30:00"`, Pydantic parses it into a standard Python `datetime` object.
* Boolean strings like `"true"`, `"1"`, `"yes"`, or `1` are coerced to `True`.

### 2. Automatic Data Validation and Error Handling
When incoming client data fails validation, Pydantic immediately catches the issue before your function code executes. 

* **Zero Boilerplate:** You don't need manual `if/else` checks or `try/except` blocks.
* **Standardized 422 Responses:** FastAPI automatically returns an `HTTP 422 Unprocessable Entity` status code with a structured, human-readable JSON error body indicating precisely which field failed and why.

#### Example 422 Validation Error Output:
```json
{
  "detail": [
    {
      "type": "int_parsing",
      "loc": ["body", "age"],
      "msg": "Input should be a valid integer, unable to parse string as an integer",
      "input": "not_a_number"
    }
  ]
}

In [24]:

uvicorn.run(app, host="127.0.0.1" , port=8001)

INFO:     Started server process [24088]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)


INFO:     127.0.0.1:60050 - "GET /search HTTP/1.1" 200 OK
INFO:     127.0.0.1:57014 - "GET /search?keyword=3 HTTP/1.1" 200 OK
INFO:     127.0.0.1:59799 - "GET /search?keyword=3&pages=54&title=3 HTTP/1.1" 200 OK
INFO:     127.0.0.1:58878 - "GET /search?keyword=3&pages=54&title=asam HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [24088]
